# Atom Training on Google Colab

Bootstrap, quick infra check, full run, and resume in one notebook.


## 1) Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Configure variables

Defaults are set for your current public repo/branch, but values are only set if missing.


In [ ]:
import os

# Repo defaults (only applied when missing)
os.environ.setdefault('ATOM_REPO_URL', 'https://github.com/MBifolco/atom.git')
os.environ.setdefault('ATOM_BRANCH', 'colab')

# Runtime/cache defaults
os.environ.setdefault('ATOM_DRIVE_REPO', '/content/drive/MyDrive/dev/atom')
os.environ.setdefault('ATOM_WORK_REPO', '/content/atom')
os.environ.setdefault('ATOM_INSTALL_JAX_CUDA', '1')
os.environ.setdefault('ATOM_JAX_VERSION', '0.7.2')
os.environ.setdefault('ATOM_DRIVE_REPO_SYNC_MODE', 'stash')  # stash|reset|skip_pull

# Auth defaults (public repo workflow)
os.environ.setdefault('ATOM_USE_GITHUB_TOKEN', '0')
if os.environ.get('ATOM_USE_GITHUB_TOKEN') != '1':
    os.environ.pop('GITHUB_TOKEN', None)

# Private repo option:
# import getpass
# os.environ['ATOM_USE_GITHUB_TOKEN'] = '1'
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ').strip()

for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
    'ATOM_DRIVE_REPO_SYNC_MODE',
    'ATOM_USE_GITHUB_TOKEN',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)


## 3) Clone (if needed) and bootstrap

This handles stale partial clones and branch checks before running `colab_bootstrap.sh`.


In [ ]:
%%bash
set -euo pipefail

WORK_REPO="${ATOM_WORK_REPO:-/content/atom}"
REPO_URL="${ATOM_REPO_URL:-}"
BRANCH="${ATOM_BRANCH:-main}"
AUTH_REPO_URL="$REPO_URL"

echo "Using WORK_REPO=$WORK_REPO"
echo "Using BRANCH=$BRANCH"

if [[ -z "$REPO_URL" ]]; then
  echo "ERROR: ATOM_REPO_URL must be set before first clone."
  exit 1
fi

if [[ "$REPO_URL" == *"<org>"* || "$REPO_URL" == *"<repo>"* ]]; then
  echo "ERROR: ATOM_REPO_URL still contains placeholders. Set a real repo URL first."
  exit 1
fi

if [[ "${ATOM_USE_GITHUB_TOKEN:-0}" == "1" && -n "${GITHUB_TOKEN:-}" && "$REPO_URL" == https://github.com/* ]]; then
  AUTH_REPO_URL="${REPO_URL/https:\/\//https:\/\/${GITHUB_TOKEN}@}"
fi

if [[ -d "$WORK_REPO" && ! -d "$WORK_REPO/.git" ]]; then
  echo "Found stale non-git directory at $WORK_REPO. Removing it..."
  rm -rf "$WORK_REPO"
fi

if [[ ! -d "$WORK_REPO/.git" ]]; then
  echo "Cloning repository..."
  if ! git clone --branch "$BRANCH" --single-branch "$AUTH_REPO_URL" "$WORK_REPO"; then
    echo "ERROR: git clone failed."
    echo "Common causes:"
    echo "  1) Wrong ATOM_REPO_URL"
    echo "  2) Branch does not exist remotely"
    echo "  3) Private repo without token"
    echo "  4) Temporary GitHub/network issue"
    exit 1
  fi
fi

cd "$WORK_REPO"
chmod +x colab_bootstrap.sh
bash colab_bootstrap.sh


## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 5) Quick infrastructure smoke test

This verifies GPU/JAX/SB3 wiring. It may not graduate curriculum with low timesteps.


In [46]:
import os
import subprocess

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "quick",
    "--device", "auto",
    "--use-vmap",
    "--output-dir", "/content/drive/MyDrive/atom_runs/quick_test",
]

proc = subprocess.Popen(
    cmd,
    cwd=work_repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
finally:
    rc = proc.wait()
print(f"\nquick smoke exit code: {rc}")


2026-03-19 22:10:56.170065: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-19 22:10:56.188573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773958256.210817   17101 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773958256.218194   17101 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773958256.237307   17101 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 6) Full training run (real run)


In [47]:
import os
import subprocess

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "complete",
    "--device", "auto",
    "--use-vmap",
    "--timesteps", "2000000",
    "--population", "8",
    "--generations", "10",
    "--output-dir", "/content/drive/MyDrive/atom_runs/run1",
]

proc = subprocess.Popen(
    cmd,
    cwd=work_repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
finally:
    rc = proc.wait()
print(f"\nfull training exit code: {rc}")
if rc != 0:
    raise RuntimeError(f"Full training failed with exit code {rc}")


2026-03-19 22:56:50.853986: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-19 22:56:50.872337: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773961010.894366   28558 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773961010.901684   28558 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773961010.920628   28558 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

RuntimeError: Full training failed with exit code 1

## 7) Resume interrupted run


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python resume_population_training.py \
  --checkpoint-dir /content/drive/MyDrive/atom_runs/run1 \
  --start-gen 8 \
  --total-gens 20 \
  --use-vmap


## Troubleshooting

- Public repo: keep `ATOM_USE_GITHUB_TOKEN=0`.
- Private repo: set `ATOM_USE_GITHUB_TOKEN=1` and provide `GITHUB_TOKEN`.
- If Drive cache has local edits, bootstrap auto-stashes by default (`ATOM_DRIVE_REPO_SYNC_MODE=stash`).
- Use `ATOM_DRIVE_REPO_SYNC_MODE=reset` to discard cache edits, or `skip_pull` to avoid pulling when dirty.
